In [ ]:
# Import packages
from os.path import join as pjoin
import pandas as pd
import numpy as np
import osgeo
import xarray as xr
import xrspatial as xrs
import rioxarray
import rasterstats as rs
import os
import glob
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import fiona
import rasterio
import re

# directory settings
input_dir = '../input'   # relative path to the input data
input_dir_ls = '../input/AGLW_1961_2021/AGLW_1961_2021'   # relative path to the input data
input_dir_crop = '../input/Crops'   # relative path to the input data
scratch_dir = '../scratch'   # relative path to the output data.

print(os.getcwd())

In [ ]:
# time bounds
years = range(2010, 2020)  # 2010 to 2019
# select South Africa (window with some margins)
xmin, xmax = 12, 35.5
ymin, ymax = -35, -22

In [ ]:
# Loading the rasterdata for all relevant years and creating a list of these rasters for MAIZE
maize_list = []
for year in years:
    path = pjoin(input_dir_crop, f'GGCP10_Production_{year}_Maize.tif')
    raster = rioxarray.open_rasterio(path)
    # Crop to South Africa bounding box
    raster = raster.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))

    # Replace nodata (-9999) with np.nan
    raster = raster.where(raster != -9999.0)
    
    # Add a new dimension for year (so they can be stacked)
    raster = raster.expand_dims(year=[year])
    maize_list.append(raster)

# Merge all years along the new 'year' dimension to create a 3D array with 10 bands (for the years).
maize_all = xr.concat(maize_list, dim='year')

# Create a new array that takes the average over years.
maize_mean = maize_all.mean(dim='year', skipna=True)

# Inspect the data in a plot: 
maize_mean.plot()

In [ ]:
# Loading the rasterdata for all relevant years and creating a list of these rasters for WHEAT
wheat_list = []
for year in years:
    path = pjoin(input_dir_crop, f'GGCP10_Production_{year}_Wheat.tif')
    raster = rioxarray.open_rasterio(path)
    # Crop to South Africa bounding box
    raster = raster.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))

    # Replace nodata (-9999) with np.nan
    raster = raster.where(raster != -9999.0)
    
    # Add a new dimension for year (so they can be stacked)
    raster = raster.expand_dims(year=[year])
    wheat_list.append(raster)

# Merge all years along the new 'year' dimension to create a 3D array with 10 bands (for the years).
wheat_all = xr.concat(wheat_list, dim='year')

# Create a new array that takes the average over years.
wheat_mean = wheat_all.mean(dim='year', skipna=True)

# Inspect the data in a plot: 
wheat_mean.plot()

In [ ]:
# Loading the rasterdata for all relevant years and creating a list of these rasters for CATTLE
cattle_list = []
for year in years:
    path = pjoin(input_dir_ls, 'Cattle', f'Cattl_{year}.tif')
    raster = rioxarray.open_rasterio(path)
    # Step 2: Crop to bounding box
    raster = raster.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
    # Add a new dimension for year (so they can be stacked)
    raster = raster.expand_dims(year=[year])
    cattle_list.append(raster)

# Merge all years along the new 'year' dimension to create a 3D array with 10 bands (for the years).
cattle_all = xr.concat(cattle_list, dim='year')

# Create a new array that takes the average over years.
cattle_mean = cattle_all.mean(dim='year')

# Inspect the data in a plot: 
cattle_mean.plot()

In [ ]:
# Loading the rasterdata for all relevant years and creating a list of these rasters for CHICKEN
chicken_list = []
for year in years:
    path = pjoin(input_dir_ls, 'Chicken', f'Chick_{year}.tif')
    raster = rioxarray.open_rasterio(path)
    # Step 2: Crop to bounding box
    raster = raster.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
    # Add a new dimension for year (so they can be stacked)
    raster = raster.expand_dims(year=[year])
    chicken_list.append(raster)

# Merge all years along the new 'year' dimension to create a 3D array with 10 bands (for the years).
chicken_all = xr.concat(chicken_list, dim='year')

# Create a new array that takes the average over years.
chicken_mean = chicken_all.mean(dim='year')

# Inspect the data in a plot: 
chicken_mean.plot()

In [ ]:
# Loading the rasterdata for all relevant years and creating a list of these rasters for SHEEP
sheep_list = []
for year in years:
    path = pjoin(input_dir_ls, 'Sheep', f'Sheep_{year}.tif')
    raster = rioxarray.open_rasterio(path)
    # Step 2: Crop to bounding box
    raster = raster.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
    # Add a new dimension for year (so they can be stacked)
    raster = raster.expand_dims(year=[year])
    sheep_list.append(raster)

# Merge all years along the new 'year' dimension to create a 3D array with 10 bands (for the years).
sheep_all = xr.concat(sheep_list, dim='year')

# Create a new array that takes the average over years.
sheep_mean = sheep_all.mean(dim='year')

# Inspect the data in a plot: 
sheep_mean.plot()

In [ ]:
# check CRS of raster data
# for livestock data:
print(sheep_mean.rio.crs)
# for crop data:
print(maize_mean.rio.crs)

In [ ]:
# get vector data for zones from e.g. municipalities
#2016 municipal boundaries
gdb = pjoin(input_dir, 'MN_2016.gdb')

fiona.listlayers(gdb)

mun16 = gpd.read_file(filename = gdb, layer='MDBLocalMunicipalBoundary2016')

# check CRS of vector data
print(mun16.crs)

In [ ]:
# calculate area of each vector:
mun16['area_km2'] = mun16['geometry'].to_crs("EPSG:6933").map(lambda p: p.area / 10**6)

In [ ]:
# Check alignment between raster and vector data
fig, ax = plt.subplots(figsize=(8, 8))
maize_mean.plot(ax=ax, cmap="viridis")
mun16.boundary.plot(ax=ax, color="red", linewidth=0.5)
plt.show()

print(cattle_mean.rio.bounds())
print(mun16.total_bounds)  

In [ ]:
# # the xarray stores the "spatial_ref" as separate dimension, so open only the usefull variable)
# cattle_mean = cattle_mean.where(cattle_mean != cattle_mean.rio.nodata)
# chicken_mean = chicken_mean.where(chicken_mean != chicken_mean.rio.nodata)
# sheep_mean = sheep_mean.where(sheep_mean != sheep_mean.rio.nodata)

cattle_mean.data

In [ ]:
# for crop data (in kilotonnes):
crop_data = {
    "Wheat": wheat_mean,
    "Maize": maize_mean
}

# for livestock data (in density):
ls_data = {
    "Cattle": cattle_mean,
    "Chicken": chicken_mean,
    "Sheep": sheep_mean,
}

# the crops can be summed to obtain the total production in each municipality (summing over the grid cells); the livestock values are averaged, 
# since this is a density value and has to be multiplied later with the vector area.
for name, variable in crop_data.items():
    raster = variable.data.squeeze()
    aff = variable.rio.transform() # The affine transformatiion scales, translates, numpy values to positions on the earth.
    stats = rs.zonal_stats(mun16.geometry, raster, affine=aff, stats=["sum"], nodata=np.nan) # generates a list of dictionaries (not so intuitive)
    mun16[f"{name}_production_total_kton"] = [s["sum"] for s in stats]
    
for name, variable in ls_data.items():
    raster = variable.data.squeeze()
    aff = variable.rio.transform() # The affine transformatiion scales, translates, numpy values to positions on the earth.
    stats = rs.zonal_stats(mun16.geometry, raster, affine=aff, stats=["mean"], nodata=np.nan) # generates a list of dictionaries (not so intuitive)
    mun16[f"{name}_density_mean"] = [s["mean"] for s in stats]

mun16

In [ ]:
mun16.explore(column='Maize_production_total_kton')

In [ ]:
mun16.explore(column='Wheat_production_total_kton')

In [ ]:
mun16.explore(column="Cattle_density_mean")

In [ ]:
mun16.explore(column="Chicken_density_mean")

In [ ]:
mun16.explore(column="Sheep_density_mean")

In [ ]:
# save to disc for inspection in QGIS or python
mun16.to_file(pjoin(scratch_dir, 'F_availability_raw_mun16.gpkg'))